# EuroCropML Benchmark - Step 4: OLMoEarth Embeddings
Load raw data, encode with OLMoEarth, train classifiers.
Uses GPU for encoding. Saves embeddings to Drive for reuse.

In [ ]:
!git clone https://github.com/mahdikalantari555/eurocrop-olmoearth-benchmark.git
%cd eurocrop-olmoearth-benchmark
!pip install -r requirements.txt -q

In [ ]:
import gc, json, os, time
import numpy as np
import torch
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/eurocrop_benchmark'

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Load raw data from local disk
from src.data.loader import load_split_padded

splits = load_split_padded(
    './preprocess', './split', 'latvia_vs_estonia',
    split_name='all', use_zenodo=True
)
X_train, y_train, _ = splits["train"]
X_test, y_test, _ = splits["test"]
del splits; gc.collect()
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Initialize OLMoEarth encoder
from src.encoder.olmoearth import OLMoEarthEncoder

encoder = OLMoEarthEncoder(
    mode="cloud",
    cloud_model_id="allenai/OlmoEarth-v1_1-Nano",
    device="cuda" if torch.cuda.is_available() else "cpu"
)

In [ ]:
# Encode and save embeddings
t = time.time()
emb_train = encoder.encode(X_train, batch_size=32)
print(f"Train embeddings: {emb_train.shape} ({time.time()-t:.1f}s)")

t = time.time()
emb_test = encoder.encode(X_test, batch_size=32)
print(f"Test embeddings: {emb_test.shape} ({time.time()-t:.1f}s)")

del X_train, X_test
gc.collect()

# Save to Drive
np.save(os.path.join(DRIVE_DIR, 'emb_train.npy'), emb_train)
np.save(os.path.join(DRIVE_DIR, 'emb_test.npy'), emb_test)
np.save(os.path.join(DRIVE_DIR, 'y_train.npy'), y_train)
np.save(os.path.join(DRIVE_DIR, 'y_test.npy'), y_test)
print("Embeddings saved to Drive")

In [ ]:
# Train classifiers on embeddings
from src.models.classical import get_classifier
from src.evaluate.metrics import compute_metrics, save_metrics, save_confusion_matrix

os.makedirs('results/metrics', exist_ok=True)
labels = sorted(set(y_test))

for clf_name in ["logreg", "rf", "lgbm", "xgb"]:
    t = time.time()
    clf = get_classifier(clf_name)
    clf.fit(emb_train, y_train)
    y_pred = clf.predict(emb_test)

    m = compute_metrics(y_test, y_pred, labels=labels)
    save_metrics(m, f'results/metrics/phase2_olmo_{clf_name}.json')
    save_confusion_matrix(y_test, y_pred, f'results/metrics/phase2_olmo_{clf_name}_cm.csv', labels=labels)
    print(f"olmo_{clf_name}: OA={m['overall_accuracy']:.3f} F1={m['macro_f1']:.3f} ({time.time()-t:.1f}s)")

    del clf; gc.collect()

In [ ]:
!cp results/metrics/phase2_*.json /content/drive/MyDrive/eurocrop_benchmark/
print("Results saved to Drive")